In [ ]:
import os
import random
from google.colab import drive
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# 1. Enforce deterministic behavior across runs
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(SEED)
  torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[+] Runtime hardware accelerator: {device}')
if torch.cuda.is_available():
  print(f'[+] GPU Device: {torch.cuda.get_device_name(0)}')

# 2. Mount Google Drive and verify directories
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/neuropeptide-classifier-benchmark/Data'
TENSORS_DIR = os.path.join(DRIVE_DIR, 'processed_tensors')
CHECKPOINTS_DIR = os.path.join(DRIVE_DIR, 'checkpoints')
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

print(f'[+] Project directory: {DRIVE_DIR}')

[+] Runtime hardware accelerator: cuda
[+] GPU Device: Tesla T4
Mounted at /content/drive
[+] Project directory: /content/drive/MyDrive/neuropeptide-classifier-benchmark/Data


In [ ]:
class MultiModalPeptideDataset(Dataset):
  """Custom Dataset pairing 1280-D ESM-2 residue tokens with 30-D biophysical descriptors."""

  def __init__(self, tokens, masks, handcrafted_df, labels):
    self.tokens = tokens.float()
    self.masks = masks.bool()
    self.handcrafted = torch.tensor(
        handcrafted_df.values, dtype=torch.float32
    )  # Expected shape: (N, 30)
    self.labels = labels.float().unsqueeze(-1)  # Shape: (N, 1)

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return {
        'tokens': self.tokens[idx],
        'mask': self.masks[idx],
        'handcrafted': self.handcrafted[idx],
        'label': self.labels[idx],
    }


def collate_multimodal_fn(batch):
  """Dynamically pads residue sequences within each batch to the batch-maximum length."""
  batch_tokens = [item['tokens'] for item in batch]
  batch_masks = [item['mask'] for item in batch]
  batch_handcrafted = torch.stack([item['handcrafted'] for item in batch])
  batch_labels = torch.stack([item['label'] for item in batch])

  max_len = max(t.size(0) for t in batch_tokens)
  d_dim = batch_tokens[0].size(1)  # 1280 for ESM-2 650M

  padded_tokens = []
  padded_masks = []
  for t, m in zip(batch_tokens, batch_masks):
    pad_len = max_len - t.size(0)
    if pad_len > 0:
      pad_t = torch.zeros((pad_len, d_dim), dtype=t.dtype)
      pad_m = torch.zeros(pad_len, dtype=m.dtype)
      padded_tokens.append(torch.cat([t, pad_t], dim=0))
      padded_masks.append(torch.cat([m, pad_m], dim=0))
    else:
      padded_tokens.append(t)
      padded_masks.append(m)

  return {
      'tokens': torch.stack(padded_tokens),
      'mask': torch.stack(padded_masks),
      'handcrafted': batch_handcrafted,
      'label': batch_labels,
  }

In [ ]:
class AttentionPooling(nn.Module):
  """Computes position-specific attention weights and aggregates residue vectors into a sequence embedding."""

  def __init__(self, hidden_dim):
    super().__init__()
    self.attn = nn.Sequential(
        nn.Linear(hidden_dim, hidden_dim // 2),
        nn.Tanh(),
        nn.Linear(hidden_dim // 2, 1),
    )

  def forward(self, x, mask=None):
    scores = self.attn(x).squeeze(-1)  # (Batch, L)
    if mask is not None:
      scores = scores.masked_fill(~mask, -1e9)
    weights = F.softmax(scores, dim=-1).unsqueeze(-1)  # (Batch, L, 1)
    pooled = torch.sum(x * weights, dim=1)  # (Batch, hidden_dim)
    return pooled, weights.squeeze(-1)


class PurePLMNet(nn.Module):
  """Model B: BiLSTM + Attention Pooling on ESM-2 650M embeddings alone."""

  def __init__(self, esm_dim=1280, lstm_hidden=128, dropout=0.3):
    super().__init__()
    self.lstm = nn.LSTM(
        input_size=esm_dim,
        hidden_size=lstm_hidden,
        bidirectional=True,
        batch_first=True,
    )
    self.pool = AttentionPooling(hidden_dim=lstm_hidden * 2)
    self.classifier = nn.Sequential(
        nn.Linear(lstm_hidden * 2, 64),
        nn.Mish(),
        nn.Dropout(dropout),
        nn.Linear(64, 1),
    )

  def forward(self, tokens, mask, **kwargs):
    lstm_out, _ = self.lstm(tokens)
    pooled, attn_weights = self.pool(lstm_out, mask)
    logits = self.classifier(pooled)
    return logits, attn_weights


class DualStreamNet(nn.Module):
  """Model C (Proposed): Dual-Stream Network with Gated Cross-Modality Fusion."""

  def __init__(
      self,
      esm_dim=1280,
      bio_dim=30,
      lstm_hidden=128,
      bio_hidden=64,
      dropout=0.3,
  ):
    super().__init__()
    # Stream A: Deep Sequence Syntax
    self.lstm = nn.LSTM(
        input_size=esm_dim,
        hidden_size=lstm_hidden,
        bidirectional=True,
        batch_first=True,
    )
    self.pool = AttentionPooling(hidden_dim=lstm_hidden * 2)

    # Stream B: Biophysical Priors (30 dense features)
    self.bio_mlp = nn.Sequential(
        nn.Linear(bio_dim, bio_hidden),
        nn.LayerNorm(bio_hidden),
        nn.Mish(),
        nn.Dropout(dropout),
        nn.Linear(bio_hidden, lstm_hidden * 2),
        nn.LayerNorm(lstm_hidden * 2),
    )

    # Gated Cross-Modality Fusion Layer
    self.gate = nn.Sequential(
        nn.Linear((lstm_hidden * 2) * 2, lstm_hidden * 2), nn.Sigmoid()
    )

    # Joint Classification Head
    self.classifier = nn.Sequential(
        nn.Linear(lstm_hidden * 2, 64),
        nn.Mish(),
        nn.Dropout(dropout),
        nn.Linear(64, 1),
    )

  def forward(self, tokens, mask, handcrafted):
    # Stream A forward pass
    lstm_out, _ = self.lstm(tokens)
    h_seq, attn_weights = self.pool(lstm_out, mask)  # (Batch, 256)

    # Stream B forward pass
    h_bio = self.bio_mlp(handcrafted)  # (Batch, 256)

    # Learned Gating: z = g * h_seq + (1 - g) * h_bio
    combined = torch.cat([h_seq, h_bio], dim=-1)
    g = self.gate(combined)
    fused = g * h_seq + (1.0 - g) * h_bio

    logits = self.classifier(fused)
    return logits, attn_weights

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5):
  """Calculates binary classification benchmark metrics."""
  y_pred = (y_prob >= threshold).astype(int)
  tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

  return {
      'Accuracy': accuracy_score(y_true, y_pred),
      'Sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0.0,
      'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
      'Precision': precision_score(y_true, y_pred, zero_division=0),
      'MCC': matthews_corrcoef(y_true, y_pred),
      'ROC-AUC': roc_auc_score(y_true, y_prob),
  }


def train_one_epoch(model, loader, optimizer, criterion):
  """Executes one training pass across the dataset."""
  model.train()
  total_loss = 0.0
  for batch in loader:
    tokens = batch['tokens'].to(device)
    mask = batch['mask'].to(device)
    handcrafted = batch['handcrafted'].to(device)
    labels = batch['label'].to(device)

    optimizer.zero_grad()
    logits, _ = model(tokens=tokens, mask=mask, handcrafted=handcrafted)
    loss = criterion(logits, labels)
    loss.backward()

    # Gradient clipping to stabilize recurrent updates
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    total_loss += loss.item()

  return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
  """Evaluates the model and returns performance metrics and predicted probabilities."""
  model.eval()
  all_probs = []
  all_labels = []

  for batch in loader:
    tokens = batch['tokens'].to(device)
    mask = batch['mask'].to(device)
    handcrafted = batch['handcrafted'].to(device)
    labels = batch['label'].cpu().numpy()

    logits, _ = model(tokens=tokens, mask=mask, handcrafted=handcrafted)
    probs = torch.sigmoid(logits).cpu().numpy()

    all_probs.extend(probs)
    all_labels.extend(labels)

  y_true = np.array(all_labels).flatten()
  y_prob = np.array(all_probs).flatten()
  return compute_metrics(y_true, y_prob), y_prob


In [ ]:
# 1. Load 650M ESM-2 representations
print('[+] Loading ESM-2 650M representations...')
train_esm = torch.load(
    os.path.join(TENSORS_DIR, 'esm2_train_representations_largeModel.pt')
)
test_esm = torch.load(os.path.join(TENSORS_DIR, 'esm2_test_representations_largeModel.pt'))

print(f"    Train tokens shape: {train_esm['tokens'].shape}")
print(f"    Test tokens shape:  {test_esm['tokens'].shape}")
assert (
    train_esm['tokens'].shape[2] == 1280
), 'Tokens do not have expected 1280 dimensions!'

# 2. Load standardized handcrafted CSVs
train_df = pd.read_csv(os.path.join(DRIVE_DIR, 'train_features_standardized.csv'))
test_df = pd.read_csv(os.path.join(DRIVE_DIR, 'test_features_standardized.csv'))

# 3. Filter down to the 30 dense biophysical features (AAC + Physicochemical)
dense_bio_cols = [
    c
    for c in train_df.columns
    if c.startswith('AAC_')
    or c
    in [
        'length',
        'molecular_weight',
        'aromaticity',
        'instability_index',
        'isoelectric_point',
        'gravy',
        'charge_pH7',
        'sec_struct_helix',
        'sec_struct_turn',
        'sec_struct_sheet',
    ]
]
print(
    f'\n[+] Isolated {len(dense_bio_cols)} dense biophysical descriptors for Stream B'
)

# 4. Construct MultiModal Dataset instances
train_dataset = MultiModalPeptideDataset(
    tokens=train_esm['tokens'],
    masks=train_esm['masks'],
    handcrafted_df=train_df[dense_bio_cols],
    labels=train_esm['labels'],
)

test_dataset = MultiModalPeptideDataset(
    tokens=test_esm['tokens'],
    masks=test_esm['masks'],
    handcrafted_df=test_df[dense_bio_cols],
    labels=test_esm['labels'],
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_multimodal_fn,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_multimodal_fn,
)

print(
    f'[+] Loaders ready: {len(train_dataset)} training samples, {len(test_dataset)} test samples.'
)

[+] Loading ESM-2 650M representations...
    Train tokens shape: torch.Size([8038, 101, 1280])
    Test tokens shape:  torch.Size([888, 100, 1280])

[+] Isolated 30 dense biophysical descriptors for Stream B
[+] Loaders ready: 8038 training samples, 888 test samples.


In [ ]:
print('=' * 75)
print('TRAINING MODEL B: Pure PLM (ESM-2 650M + BiLSTM + Attention)')
print('=' * 75)

model_b = PurePLMNet(esm_dim=1280, lstm_hidden=128, dropout=0.3).to(device)
optimizer_b = torch.optim.AdamW(
    model_b.parameters(), lr=5e-4, weight_decay=1e-2
)
scheduler_b = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_b, T_max=15)
criterion = nn.BCEWithLogitsLoss()

best_mcc_b = -1.0
best_metrics_b = None
checkpoint_path_b = os.path.join(CHECKPOINTS_DIR, 'best_pure_plm_650M.pt')

for epoch in range(1, 16):
  train_loss = train_one_epoch(model_b, train_loader, optimizer_b, criterion)
  scheduler_b.step()
  val_metrics, _ = evaluate(model_b, test_loader)

  print(
      f"Epoch {epoch:02d}/15 | Loss: {train_loss:.4f} | ACC: {val_metrics['Accuracy']:.4f} | "
      f"SN: {val_metrics['Sensitivity']:.4f} | SP: {val_metrics['Specificity']:.4f} | "
      f"MCC: {val_metrics['MCC']:.4f} | AUC: {val_metrics['ROC-AUC']:.4f}"
  )

  if val_metrics['MCC'] > best_mcc_b:
    best_mcc_b = val_metrics['MCC']
    best_metrics_b = val_metrics
    torch.save(model_b.state_dict(), checkpoint_path_b)

best_metrics_b['Model'] = 'Model B: Pure PLM (ESM-2 650M + BiLSTM)'
best_metrics_b['Evaluation'] = 'Independent Test'
print(f'\n[+] Best Model B checkpoint saved to: {checkpoint_path_b}')


TRAINING MODEL B: Pure PLM (ESM-2 650M + BiLSTM + Attention)
Epoch 01/15 | Loss: 0.3488 | ACC: 0.9077 | SN: 0.9189 | SP: 0.8964 | MCC: 0.8155 | AUC: 0.9627
Epoch 02/15 | Loss: 0.2075 | ACC: 0.8930 | SN: 0.8446 | SP: 0.9414 | MCC: 0.7897 | AUC: 0.9670
Epoch 03/15 | Loss: 0.1929 | ACC: 0.9099 | SN: 0.9730 | SP: 0.8468 | MCC: 0.8264 | AUC: 0.9681
Epoch 04/15 | Loss: 0.1710 | ACC: 0.9212 | SN: 0.9234 | SP: 0.9189 | MCC: 0.8424 | AUC: 0.9697
Epoch 05/15 | Loss: 0.1475 | ACC: 0.9144 | SN: 0.9212 | SP: 0.9077 | MCC: 0.8289 | AUC: 0.9694
Epoch 06/15 | Loss: 0.1306 | ACC: 0.9155 | SN: 0.9167 | SP: 0.9144 | MCC: 0.8311 | AUC: 0.9677
Epoch 07/15 | Loss: 0.1161 | ACC: 0.9122 | SN: 0.9257 | SP: 0.8986 | MCC: 0.8246 | AUC: 0.9677
Epoch 08/15 | Loss: 0.1020 | ACC: 0.9178 | SN: 0.9009 | SP: 0.9347 | MCC: 0.8361 | AUC: 0.9692
Epoch 09/15 | Loss: 0.0861 | ACC: 0.9155 | SN: 0.9234 | SP: 0.9077 | MCC: 0.8312 | AUC: 0.9676
Epoch 10/15 | Loss: 0.0747 | ACC: 0.9234 | SN: 0.9144 | SP: 0.9324 | MCC: 0.8470 | A

In [ ]:
print('=' * 75)
print('TRAINING MODEL C: Dual-Stream Gated Network (650M + Biophysical Priors)')
print('=' * 75)

model_c = DualStreamNet(
    esm_dim=1280,
    bio_dim=len(dense_bio_cols),
    lstm_hidden=128,
    bio_hidden=64,
    dropout=0.3,
).to(device)

optimizer_c = torch.optim.AdamW(
    model_c.parameters(), lr=4e-4, weight_decay=1e-2
)
scheduler_c = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_c, T_max=15)

best_mcc_c = -1.0
best_metrics_c = None
checkpoint_path_c = os.path.join(CHECKPOINTS_DIR, 'best_dual_stream_650M.pt')

for epoch in range(1, 16):
  train_loss = train_one_epoch(model_c, train_loader, optimizer_c, criterion)
  scheduler_c.step()
  val_metrics, _ = evaluate(model_c, test_loader)

  print(
      f"Epoch {epoch:02d}/15 | Loss: {train_loss:.4f} | ACC: {val_metrics['Accuracy']:.4f} | "
      f"SN: {val_metrics['Sensitivity']:.4f} | SP: {val_metrics['Specificity']:.4f} | "
      f"MCC: {val_metrics['MCC']:.4f} | AUC: {val_metrics['ROC-AUC']:.4f}"
  )

  if val_metrics['MCC'] > best_mcc_c:
    best_mcc_c = val_metrics['MCC']
    best_metrics_c = val_metrics
    torch.save(model_c.state_dict(), checkpoint_path_c)

best_metrics_c['Model'] = (
    'Model C: Dual-Stream Gated Network (650M + Biophysics)'
)
best_metrics_c['Evaluation'] = 'Independent Test'
print(f'\n[+] Best Model C checkpoint saved to: {checkpoint_path_c}')

TRAINING MODEL C: Dual-Stream Gated Network (650M + Biophysical Priors)
Epoch 01/15 | Loss: 0.4182 | ACC: 0.8998 | SN: 0.9234 | SP: 0.8761 | MCC: 0.8004 | AUC: 0.9581
Epoch 02/15 | Loss: 0.2286 | ACC: 0.8975 | SN: 0.8716 | SP: 0.9234 | MCC: 0.7961 | AUC: 0.9653
Epoch 03/15 | Loss: 0.1891 | ACC: 0.9032 | SN: 0.9032 | SP: 0.9032 | MCC: 0.8063 | AUC: 0.9664
Epoch 04/15 | Loss: 0.1693 | ACC: 0.8998 | SN: 0.9144 | SP: 0.8851 | MCC: 0.7999 | AUC: 0.9640
Epoch 05/15 | Loss: 0.1537 | ACC: 0.8964 | SN: 0.8649 | SP: 0.9279 | MCC: 0.7944 | AUC: 0.9667
Epoch 06/15 | Loss: 0.1332 | ACC: 0.9178 | SN: 0.9302 | SP: 0.9054 | MCC: 0.8358 | AUC: 0.9682
Epoch 07/15 | Loss: 0.1193 | ACC: 0.9065 | SN: 0.8986 | SP: 0.9144 | MCC: 0.8132 | AUC: 0.9675
Epoch 08/15 | Loss: 0.1022 | ACC: 0.9043 | SN: 0.9032 | SP: 0.9054 | MCC: 0.8086 | AUC: 0.9655
Epoch 09/15 | Loss: 0.0879 | ACC: 0.9099 | SN: 0.9009 | SP: 0.9189 | MCC: 0.8200 | AUC: 0.9656
Epoch 10/15 | Loss: 0.0745 | ACC: 0.9065 | SN: 0.8941 | SP: 0.9189 | MCC:

In [ ]:
# Model A: Your verified Day 3 Handcrafted Baseline (SVM RBF)
model_a = {
    'Model': 'Model A: Handcrafted ML (SVM RBF)',
    'Evaluation': 'Independent Test',
    'Accuracy': 0.9009,
    'Sensitivity': 0.9122,
    'Specificity': 0.8896,
    'Precision': 0.8921,
    'MCC': 0.8020,
    'ROC-AUC': 0.9547,
}

# Compile ablation results
ablation_df = pd.DataFrame([model_a, best_metrics_b, best_metrics_c])
col_order = [
    'Model',
    'Evaluation',
    'Accuracy',
    'Sensitivity',
    'Specificity',
    'Precision',
    'MCC',
    'ROC-AUC',
]
ablation_df = ablation_df[col_order].round(4)

print('\n' + '=' * 80)
print('FINAL TRIPARTITE ABLATION BENCHMARK (ESM-2 650M BACKBONE)')
print('=' * 80)
print(ablation_df.to_string(index=False))

# Save table to Google Drive
output_csv_path = os.path.join(DRIVE_DIR, 'ablation_benchmark_results_650M.csv')
ablation_df.to_csv(output_csv_path, index=False)
print(f'\n[+] Final benchmark table saved to: {output_csv_path}')


FINAL TRIPARTITE ABLATION BENCHMARK (ESM-2 650M BACKBONE)
                                                 Model       Evaluation  Accuracy  Sensitivity  Specificity  Precision    MCC  ROC-AUC
                     Model A: Handcrafted ML (SVM RBF) Independent Test    0.9009       0.9122       0.8896     0.8921 0.8020   0.9547
               Model B: Pure PLM (ESM-2 650M + BiLSTM) Independent Test    0.9234       0.9144       0.9324     0.9312 0.8470   0.9692
Model C: Dual-Stream Gated Network (650M + Biophysics) Independent Test    0.9178       0.9302       0.9054     0.9077 0.8358   0.9682

[+] Final benchmark table saved to: /content/drive/MyDrive/neuropeptide-classifier-benchmark/Data/ablation_benchmark_results_650M.csv
